# Web Development with Flask

**Flask** is a lightweight WSGI web framework for Python. It's minimal, flexible, and great for building APIs and small-to-medium web applications.

Install: `pip install flask`

**Note**: Flask apps run as a server. In this notebook we demonstrate the code patterns. To run a Flask app, save the code to a `.py` file and run it with `python app.py`.

In [ ]:
import flask
print(f'Flask version: {flask.__version__}')

## 1. The Simplest Flask App

In [ ]:
# The smallest possible Flask app
# Save this to app.py and run: python app.py

app_code = '''
from flask import Flask

app = Flask(__name__)

@app.route('/')  # decorator maps URL to function
def home():
    return '<h1>Hello, World!</h1>'

if __name__ == '__main__':
    app.run(debug=True, port=5000)
'''

print(app_code)
print('Access at: http://127.0.0.1:5000/')

## 2. Routes and URL Variables

In [ ]:
routes_code = '''
from flask import Flask

app = Flask(__name__)

@app.route('/')
def home():
    return 'Home Page'

@app.route('/about')
def about():
    return 'About Page'

# URL variable: <username>
@app.route('/user/<username>')
def user_profile(username):
    return f'Profile: {username}'

# Typed URL variable: <int:id>
@app.route('/post/<int:post_id>')
def show_post(post_id):
    return f'Post #{post_id}'

# Multiple routes for one function
@app.route('/hello')
@app.route('/hello/<name>')
def hello(name="World"):
    return f'Hello, {name}!'
'''

print(routes_code)

## 3. HTTP Methods: GET and POST

In [ ]:
methods_code = '''
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route('/submit', methods=['GET', 'POST'])
def submit():
    if request.method == 'POST':
        # Get form data
        name = request.form.get('name')
        email = request.form.get('email')
        return f'Received: {name}, {email}'
    else:
        # Show form
        return \'\'\'<form method="POST">
            <input name="name" placeholder="Name">
            <input name="email" placeholder="Email">
            <button type="submit">Submit</button>
        </form>\'\'\'

# JSON endpoint
@app.route('/api/data', methods=['POST'])
def receive_json():
    data = request.get_json()  # parse JSON body
    name = data.get(\'name\', \'unknown\')
    return jsonify({\'status\': \'ok\', \'received\': name})
'''

print(methods_code)

## 4. Building a REST API with Flask

Let's build a complete in-memory To-Do list API.

In [ ]:
# Complete Flask REST API (save to todo_api.py and run)
todo_api = '''
from flask import Flask, jsonify, request, abort

app = Flask(__name__)

# In-memory storage
todos = [
    {\'id\': 1, \'title\': \'Buy groceries\', \'done\': False},
    {\'id\': 2, \'title\': \'Learn Flask\',   \'done\': True},
]
next_id = 3

def find_todo(todo_id):
    return next((t for t in todos if t[\'id\'] == todo_id), None)

# GET /todos — list all
@app.route(\'/todos\', methods=[\'GET\'])
def get_todos():
    done_filter = request.args.get(\'done\')  # ?done=true
    result = todos
    if done_filter is not None:
        result = [t for t in todos if str(t[\'done\']).lower() == done_filter]
    return jsonify(result)

# GET /todos/<id> — get one
@app.route(\'/todos/<int:todo_id>\', methods=[\'GET\'])
def get_todo(todo_id):
    todo = find_todo(todo_id)
    if not todo:
        abort(404, description=f\'Todo {todo_id} not found\')
    return jsonify(todo)

# POST /todos — create
@app.route(\'/todos\', methods=[\'POST\'])
def create_todo():
    global next_id
    data = request.get_json()
    if not data or not data.get(\'title\'):
        abort(400, description=\'title is required\')
    todo = {\'id\': next_id, \'title\': data[\'title\'], \'done\': data.get(\'done\', False)}
    todos.append(todo)
    next_id += 1
    return jsonify(todo), 201

# PUT /todos/<id> — update
@app.route(\'/todos/<int:todo_id>\', methods=[\'PUT\'])
def update_todo(todo_id):
    todo = find_todo(todo_id)
    if not todo:
        abort(404)
    data = request.get_json()
    todo.update({k: v for k, v in data.items() if k in (\'title\', \'done\')})
    return jsonify(todo)

# DELETE /todos/<id>
@app.route(\'/todos/<int:todo_id>\', methods=[\'DELETE\'])
def delete_todo(todo_id):
    todo = find_todo(todo_id)
    if not todo:
        abort(404)
    todos.remove(todo)
    return \'\', 204

@app.errorhandler(404)
def not_found(e):
    return jsonify({\'error\': str(e)}), 404

@app.errorhandler(400)
def bad_request(e):
    return jsonify({\'error\': str(e)}), 400

if __name__ == \'__main__\':
    app.run(debug=True)
'''

print('Flask To-Do API code:')
print(todo_api[:500], '...')

## 5. Testing a Flask App

Flask includes a **test client** so you can test your app without running a server.

In [ ]:
from flask import Flask, jsonify, request, abort

# Create the app in test mode
app = Flask(__name__)
app.config['TESTING'] = True

items = [{'id': 1, 'name': 'Apple'}, {'id': 2, 'name': 'Banana'}]

@app.route('/items')
def get_items():
    return jsonify(items)

@app.route('/items/<int:item_id>')
def get_item(item_id):
    item = next((i for i in items if i['id'] == item_id), None)
    if not item:
        abort(404)
    return jsonify(item)

@app.route('/items', methods=['POST'])
def create_item():
    data = request.get_json()
    new_item = {'id': len(items) + 1, 'name': data['name']}
    items.append(new_item)
    return jsonify(new_item), 201

# Test using the test client
with app.test_client() as client:
    # GET all items
    r = client.get('/items')
    print(f'GET /items: {r.status_code}')
    print(f'  Data: {r.get_json()}')
    
    # GET one item
    r = client.get('/items/1')
    print(f'\nGET /items/1: {r.status_code}')
    print(f'  Data: {r.get_json()}')
    
    # GET non-existent
    r = client.get('/items/99')
    print(f'\nGET /items/99: {r.status_code}')  # 404
    
    # POST new item
    r = client.post('/items', json={'name': 'Cherry'})
    print(f'\nPOST /items: {r.status_code}')
    print(f'  Created: {r.get_json()}')

## 6. Blueprints — Organizing Large Apps

In [ ]:
blueprint_code = '''
# users.py — a blueprint for user-related routes
from flask import Blueprint, jsonify

users_bp = Blueprint(\'users\', __name__, url_prefix=\'/users\')

@users_bp.route(\'/')
def list_users():
    return jsonify([\'alice\', \'bob\'])

@users_bp.route(\'/<username>\')
def get_user(username):
    return jsonify({\'user\': username})


# app.py — main app registers blueprints
from flask import Flask
from users import users_bp

app = Flask(__name__)
app.register_blueprint(users_bp)

if __name__ == \'__main__\':
    app.run(debug=True)
'''

print('Blueprint structure:')
print(blueprint_code)

## Mini-Project: In-Memory REST API (Fully Tested)

In [ ]:
from flask import Flask, jsonify, request, abort

app = Flask(__name__)
app.config['TESTING'] = True

# In-memory book store
books = [
    {'id': 1, 'title': 'Clean Code', 'author': 'Robert Martin', 'available': True},
    {'id': 2, 'title': 'Python Cookbook', 'author': 'David Beazley', 'available': True},
]
book_counter = [3]

def find_book(book_id):
    return next((b for b in books if b['id'] == book_id), None)

@app.route('/books')
def list_books():
    available = request.args.get('available')
    result = books
    if available is not None:
        result = [b for b in books if str(b['available']).lower() == available]
    return jsonify({'books': result, 'total': len(result)})

@app.route('/books/<int:book_id>')
def get_book(book_id):
    book = find_book(book_id)
    if not book: abort(404)
    return jsonify(book)

@app.route('/books', methods=['POST'])
def add_book():
    data = request.get_json()
    if not data or not data.get('title'):
        abort(400)
    book = {'id': book_counter[0], 'title': data['title'],
            'author': data.get('author', 'Unknown'), 'available': True}
    books.append(book)
    book_counter[0] += 1
    return jsonify(book), 201

@app.route('/books/<int:book_id>/checkout', methods=['POST'])
def checkout(book_id):
    book = find_book(book_id)
    if not book: abort(404)
    if not book['available']:
        return jsonify({'error': 'Book not available'}), 400
    book['available'] = False
    return jsonify({'message': f"Checked out '{book['title']}"})

@app.route('/books/<int:book_id>/return', methods=['POST'])
def return_book(book_id):
    book = find_book(book_id)
    if not book: abort(404)
    book['available'] = True
    return jsonify({'message': f"Returned '{book['title']}"})

@app.errorhandler(404)
def not_found(e):
    return jsonify({'error': 'Not found'}), 404

@app.errorhandler(400)
def bad_request(e):
    return jsonify({'error': 'Bad request'}), 400


# Test the API
with app.test_client() as client:
    print('=== Library API Tests ===')
    
    r = client.get('/books')
    print(f'All books: {r.get_json()["total"]} books')
    
    r = client.post('/books', json={'title': 'Fluent Python', 'author': 'Luciano Ramalho'})
    print(f'Added: {r.get_json()["title"]} (id={r.get_json()["id"]})')
    
    r = client.post('/books/1/checkout')
    print(f'Checkout: {r.get_json()["message"]}')
    
    r = client.post('/books/1/checkout')
    print(f'Re-checkout: {r.get_json()["error"]} (status={r.status_code})')
    
    r = client.get('/books?available=true')
    available = r.get_json()['books']
    print(f'Available books: {[b["title"] for b in available]}')